In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile
import xml.etree.ElementTree as ET
import re
import json

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
EARTHQUAKE_DIR = RAW_DIR / "earthquakes"
SHAKEMAP_DIR = RAW_DIR / "shakemap"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
earthquakes = pd.read_csv(
    EARTHQUAKE_DIR /
    "india_region_earthquakes_2000_2026.csv"
)

earthquakes["time"] = pd.to_datetime(
    earthquakes["time"],
    format="mixed",
    utc=True
)

print(earthquakes.shape)
earthquakes.head()

(18339, 11)


,time,latitude,longitude,depth,mag,magType,place,type,status,net,id
0,2000-01-01 05:24:35.290000+00:00,36.874,69.947,54.3,5.1,mwc,"29 km SSE of Rust?q, Afghanistan",earthquake,reviewed,us,usp0009kk8
1,2000-01-01 06:26:04.210000+00:00,37.027,69.964,33.0,4.4,mb,"16 km SE of Rust?q, Afghanistan",earthquake,reviewed,us,usp0009kkd
2,2000-01-02 10:21:17.550000+00:00,36.229,70.893,33.0,4.1,mb,"70 km S of Jurm, Afghanistan",earthquake,reviewed,us,usp0009kn4
3,2000-01-02 10:23:58.980000+00:00,27.559,92.498,33.0,5.0,mb,"33 km NNE of Bomdila, India",earthquake,reviewed,us,usp0009kn5
4,2000-01-03 22:34:12.640000+00:00,22.132,92.771,33.0,4.6,mb,"45 km SSW of Saiha, India",earthquake,reviewed,us,usp0009kqm


In [5]:
product_inventory = pd.read_csv(
    SHAKEMAP_DIR /
    "shakemap_product_inventory.csv"
)

print(product_inventory.shape)
product_inventory.head()

(40, 8)


,event_id,update_time,source,version,url,filename,size_bytes,size_MB
0,usc000ff4h,1364340210594,us,NaN,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip,NaN,NaN
1,usb000fzn7,1367582401348,us,NaN,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip,NaN,NaN
2,usb000g112,1366372786546,us,NaN,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip,NaN,NaN
3,usb000hdu8,1371128314208,us,NaN,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip,283341.0,0.270215
4,usc000kmdj,1388709976346,us,NaN,https://earthquake.usgs.gov/product/shakemap/u...,download/grid.xyz.zip,279431.0,0.266486


In [6]:
downloaded_files = []

for event_dir in SHAKEMAP_DIR.iterdir():

    if not event_dir.is_dir():
        continue

    for file in event_dir.iterdir():

        if file.is_file():

            downloaded_files.append({
                "event_id": event_dir.name,
                "file": file
            })

downloaded_files = pd.DataFrame(
    downloaded_files
)

downloaded_files

,event_id,file
0,us10002mft,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
1,us10002n5q,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
2,us100030qs,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
3,us10003vpz,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
4,us10003vry,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
5,us10003vsp,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
6,us10003vxc,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
7,us100042n2,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
8,us10004dtm,C:\Documents\Earthquake_ML_DL\data\raw\shakema...
9,us10004rhs,C:\Documents\Earthquake_ML_DL\data\raw\shakema...


In [7]:
downloaded_files["suffix"] = (
    downloaded_files["file"]
    .apply(lambda x: x.suffix.lower())
)

downloaded_files["suffix"].value_counts()

suffix
.zip    40
Name: count, dtype: int64

In [8]:
sample_file = downloaded_files.iloc[0]["file"]

print("File:")
print(sample_file)

with zipfile.ZipFile(sample_file, "r") as z:

    print("\nContents:")
    
    for name in z.namelist():
        print(name)

File:
C:\Documents\Earthquake_ML_DL\data\raw\shakemap\us10002mft\grid.xyz.zip

Contents:
grid.xyz


In [9]:
with zipfile.ZipFile(sample_file, "r") as z:

    names = z.namelist()

    grid_files = [
        name for name in names
        if "grid.xyz" in name.lower()
    ]

    print(grid_files)

['grid.xyz']


In [10]:
with zipfile.ZipFile(sample_file, "r") as z:

    grid_name = grid_files[0]

    with z.open(grid_name) as f:

        lines = [
            line.decode(
                "utf-8",
                errors="ignore"
            )
            for line in f
        ]

print("\n".join(lines[:30]))

us10002mft 5.5 36.68 71.30 JUN 29 2015 22:07:48 UTC 69.3004 35.0762 73.3004 38.2842 (Process time: Sun Aug 09 21:37:32 2015) HINDU KUSH REGION, AFGHANISTAN

69.3004 38.2842 0.4 0.17 2.54 0.71 0.18 0.03

69.3171 38.2842 0.41 0.18 2.57 0.74 0.19 0.03

69.3337 38.2842 0.4 0.16 2.53 0.71 0.17 0.03

69.3504 38.2842 0.38 0.15 2.46 0.68 0.16 0.03

69.3671 38.2842 0.4 0.16 2.5 0.71 0.17 0.03

69.3837 38.2842 0.39 0.15 2.49 0.7 0.16 0.03

69.4004 38.2842 0.34 0.11 2.29 0.6 0.12 0.02

69.4171 38.2842 0.34 0.12 2.31 0.61 0.12 0.02

69.4337 38.2842 0.36 0.13 2.35 0.65 0.13 0.02

69.4504 38.2842 0.36 0.12 2.33 0.63 0.13 0.02

69.4671 38.2842 0.35 0.12 2.31 0.63 0.12 0.02

69.4837 38.2842 0.39 0.14 2.45 0.7 0.15 0.03

69.5004 38.2842 0.44 0.17 2.56 0.78 0.18 0.03

69.5171 38.2842 0.42 0.16 2.51 0.75 0.17 0.03

69.5337 38.2842 0.44 0.17 2.55 0.78 0.18 0.03

69.5504 38.2842 0.45 0.17 2.56 0.79 0.18 0.03

69.5671 38.2842 0.46 0.18 2.6 0.82 0.19 0.03

69.5837 38.2842 0.5 0.21 2.68 0.88 0.22 0.04

69.600

In [17]:
def parse_grid_xyz(zip_path):

    with zipfile.ZipFile(zip_path, "r") as z:

        # Find grid.xyz
        xyz_files = [
            name for name in z.namelist()
            if name.lower().endswith("grid.xyz")
        ]

        if not xyz_files:
            raise FileNotFoundError(
                f"No grid.xyz found in {zip_path}"
            )

        grid_name = xyz_files[0]

        with z.open(grid_name) as f:

            lines = [
                line.decode(
                    "utf-8",
                    errors="replace"
                ).strip()
                for line in f
            ]

    # ---------------------------------------------------------
    # First line = ShakeMap metadata
    # ---------------------------------------------------------

    metadata_line = lines[0]

    parts = metadata_line.split()

    event_id = parts[0]
    magnitude = float(parts[1])
    event_lat = float(parts[2])
    event_lon = float(parts[3])

    # ---------------------------------------------------------
    # Ground-motion data
    # ---------------------------------------------------------

    records = []

    for line in lines[1:]:

        if not line:
            continue

        values = line.split()

        # We expect:
        # lon lat PGA PGV MMI PSA03 PSA10 PSA30

        if len(values) < 8:
            continue

        try:

            records.append([
                float(values[0]),  # longitude
                float(values[1]),  # latitude
                float(values[2]),  # PGA
                float(values[3]),  # PGV
                float(values[4]),  # MMI
                float(values[5]),  # PSA03
                float(values[6]),  # PSA10
                float(values[7])   # PSA30
            ])

        except ValueError:
            continue

    grid = pd.DataFrame(
        records,
        columns=[
            "grid_lon",
            "grid_lat",
            "PGA",
            "PGV",
            "MMI",
            "PSA03",
            "PSA10",
            "PSA30"
        ]
    )

    # ---------------------------------------------------------
    # Add event metadata
    # ---------------------------------------------------------

    grid["event_id"] = event_id
    grid["grid_magnitude"] = magnitude
    grid["grid_event_lat"] = event_lat
    grid["grid_event_lon"] = event_lon

    return grid

In [18]:
sample_grid = parse_grid_xyz(
    sample_file
)

print("Shape:", sample_grid.shape)

print("\nColumns:")
print(sample_grid.columns.tolist())

print("\nFirst 5 rows:")
display(sample_grid.head())

Shape: (46513, 12)

Columns:
['grid_lon', 'grid_lat', 'PGA', 'PGV', 'MMI', 'PSA03', 'PSA10', 'PSA30', 'event_id', 'grid_magnitude', 'grid_event_lat', 'grid_event_lon']

First 5 rows:


,grid_lon,grid_lat,PGA,PGV,MMI,PSA03,PSA10,PSA30,event_id,grid_magnitude,grid_event_lat,grid_event_lon
0,69.3004,38.2842,0.40,0.17,2.54,0.71,0.18,0.03,us10002mft,5.5,36.68,71.3
1,69.3171,38.2842,0.41,0.18,2.57,0.74,0.19,0.03,us10002mft,5.5,36.68,71.3
2,69.3337,38.2842,0.40,0.16,2.53,0.71,0.17,0.03,us10002mft,5.5,36.68,71.3
3,69.3504,38.2842,0.38,0.15,2.46,0.68,0.16,0.03,us10002mft,5.5,36.68,71.3
4,69.3671,38.2842,0.40,0.16,2.50,0.71,0.17,0.03,us10002mft,5.5,36.68,71.3


In [19]:
sample_grid.describe()

,grid_lon,grid_lat,PGA,PGV,MMI,PSA03,PSA10,PSA30,grid_magnitude,grid_event_lat,grid_event_lon
count,46513.000000,46513.000000,46513.000000,46513.000000,46513.000000,46513.000000,46513.000000,46513.000000,46513.0,46513.00,4.651300e+04
mean,71.300400,36.680201,1.167381,0.291073,2.848082,1.917425,0.307492,0.045191,5.5,36.68,7.130000e+01
std,1.159514,0.930890,0.566890,0.107929,0.234869,0.870974,0.114016,0.014218,0.0,0.00,1.421101e-14
min,69.300400,35.076200,0.310000,0.110000,2.250000,0.550000,0.110000,0.020000,5.5,36.68,7.130000e+01
25%,70.300400,35.878200,0.730000,0.210000,2.670000,1.240000,0.220000,0.030000,5.5,36.68,7.130000e+01
50%,71.300400,36.680200,1.010000,0.270000,2.840000,1.690000,0.280000,0.040000,5.5,36.68,7.130000e+01
75%,72.300400,37.482200,1.510000,0.360000,3.030000,2.460000,0.380000,0.050000,5.5,36.68,7.130000e+01
max,73.300400,38.284200,3.290000,0.870000,3.590000,5.190000,0.910000,0.120000,5.5,36.68,7.130000e+01


In [20]:
print("Unique event:", sample_grid["event_id"].unique())

print(
    "\nPGA range:",
    sample_grid["PGA"].min(),
    "→",
    sample_grid["PGA"].max()
)

print(
    "PGV range:",
    sample_grid["PGV"].min(),
    "→",
    sample_grid["PGV"].max()
)

print(
    "MMI range:",
    sample_grid["MMI"].min(),
    "→",
    sample_grid["MMI"].max()
)

Unique event: <StringArray>
['us10002mft']
Length: 1, dtype: str

PGA range: 0.31 → 3.29
PGV range: 0.11 → 0.87
MMI range: 2.25 → 3.59


In [21]:
sample_grid.isna().sum()

grid_lon          0
grid_lat          0
PGA               0
PGV               0
MMI               0
PSA03             0
PSA10             0
PSA30             0
event_id          0
grid_magnitude    0
grid_event_lat    0
grid_event_lon    0
dtype: int64

In [23]:
from tqdm.notebook import tqdm

In [24]:
all_grids = []
failed_events = []

for _, row in tqdm(
    downloaded_files.iterrows(),
    total=len(downloaded_files),
    desc="Processing ShakeMap grids"
):

    event_id = row["event_id"]
    file_path = row["file"]

    try:
        grid = parse_grid_xyz(file_path)
        all_grids.append(grid)

    except Exception as e:
        failed_events.append({
            "event_id": event_id,
            "error": str(e)
        })

print("Successfully processed:", len(all_grids))
print("Failed:", len(failed_events))

Processing ShakeMap grids:   0%|          | 0/40 [00:00<?, ?it/s]

Successfully processed: 40
Failed: 0


In [25]:
ground_motion = pd.concat(
    all_grids,
    ignore_index=True
)

print("Ground-motion dataset shape:", ground_motion.shape)
display(ground_motion.head())

Ground-motion dataset shape: (2912739, 12)


,grid_lon,grid_lat,PGA,PGV,MMI,PSA03,PSA10,PSA30,event_id,grid_magnitude,grid_event_lat,grid_event_lon
0,69.3004,38.2842,0.40,0.17,2.54,0.71,0.18,0.03,us10002mft,5.5,36.68,71.3
1,69.3171,38.2842,0.41,0.18,2.57,0.74,0.19,0.03,us10002mft,5.5,36.68,71.3
2,69.3337,38.2842,0.40,0.16,2.53,0.71,0.17,0.03,us10002mft,5.5,36.68,71.3
3,69.3504,38.2842,0.38,0.15,2.46,0.68,0.16,0.03,us10002mft,5.5,36.68,71.3
4,69.3671,38.2842,0.40,0.16,2.50,0.71,0.17,0.03,us10002mft,5.5,36.68,71.3


In [26]:
print("Total ground-motion observations:", f"{len(ground_motion):,}")
print("Unique earthquake events:", ground_motion["event_id"].nunique())

print("\nObservations per event:")
print(
    ground_motion["event_id"]
    .value_counts()
    .describe()
)

Total ground-motion observations: 2,912,739
Unique earthquake events: 40

Observations per event:
count        40.000000
mean      72818.475000
std       50548.553092
min       25277.000000
25%       46754.000000
50%       56514.500000
75%       57840.000000
max      228956.000000
Name: count, dtype: float64


In [27]:
print("\nColumns:")
print(ground_motion.columns.tolist())


Columns:
['grid_lon', 'grid_lat', 'PGA', 'PGV', 'MMI', 'PSA03', 'PSA10', 'PSA30', 'event_id', 'grid_magnitude', 'grid_event_lat', 'grid_event_lon']


In [28]:
print("\nMissing values:")
display(ground_motion.isna().sum())


Missing values:


grid_lon          0
grid_lat          0
PGA               0
PGV               0
MMI               0
PSA03             0
PSA10             0
PSA30             0
event_id          0
grid_magnitude    0
grid_event_lat    0
grid_event_lon    0
dtype: int64

In [29]:
print("PGA:")
display(ground_motion["PGA"].describe())

print("\nPGV:")
display(ground_motion["PGV"].describe())

print("\nMMI:")
display(ground_motion["MMI"].describe())

print("\nPSA03:")
display(ground_motion["PSA03"].describe())

print("\nPSA10:")
display(ground_motion["PSA10"].describe())

print("\nPSA30:")
display(ground_motion["PSA30"].describe())

PGA:


count    2.912739e+06
mean     5.967489e-01
std      1.239956e+00
min      0.000000e+00
25%      1.000000e-01
50%      2.800000e-01
75%      6.900000e-01
max      9.017000e+01
Name: PGA, dtype: float64


PGV:


count    2.912739e+06
mean     3.989468e-01
std      7.397071e-01
min      1.000000e-02
25%      1.000000e-01
50%      2.100000e-01
75%      4.600000e-01
max      4.144000e+01
Name: PGV, dtype: float64


MMI:


...

count    2.912739e+06
mean     1.226700e+00
std      2.264547e+00
min      1.000000e-02
25%      2.600000e-01
50%      6.800000e-01
75%      1.500000e+00
max      1.391900e+02
Name: PSA03, dtype: float64


PSA10:


count    2.912739e+06
mean     4.418295e-01
std      7.489411e-01
min      1.000000e-02
25%      1.300000e-01
50%      2.400000e-01
75%      5.100000e-01
max      4.552000e+01
Name: PSA10, dtype: float64


PSA30:


count    2.912739e+06
mean     9.818873e-02
std      1.589612e-01
min      0.000000e+00
25%      3.000000e-02
50%      5.000000e-02
75%      1.000000e-01
max      7.700000e+00
Name: PSA30, dtype: float64

In [33]:
source_features = (
    ground_motion[
        [
            "event_id",
            "grid_magnitude",
            "grid_event_lat",
            "grid_event_lon"
        ]
    ]
    .drop_duplicates("event_id")
    .copy()
)

source_features = source_features.rename(
    columns={
        "grid_magnitude": "magnitude",
        "grid_event_lat": "event_lat",
        "grid_event_lon": "event_lon"
    }
)

print("Source events:", len(source_features))

display(source_features.head())

Source events: 40


,event_id,magnitude,event_lat,event_lon
0,us10002mft,5.5,36.68,71.30
46513,us10002n5q,5.5,11.42,95.03
103389,us100030qs,5.9,36.53,71.21
150143,us10003vpz,5.5,6.88,94.55
207742,us10003vry,6.6,6.84,94.65


In [34]:
dataset = ground_motion.copy()

print("Rows:", len(dataset))

print(
    "Missing magnitude before merge:",
    dataset["grid_magnitude"].isna().sum()
)

# Rename the metadata already extracted from grid.xyz
dataset = dataset.rename(
    columns={
        "grid_magnitude": "magnitude",
        "grid_event_lat": "event_lat",
        "grid_event_lon": "event_lon"
    }
)

print(
    "Missing magnitude after:",
    dataset["magnitude"].isna().sum()
)

print(
    "Missing event latitude:",
    dataset["event_lat"].isna().sum()
)

print(
    "Missing event longitude:",
    dataset["event_lon"].isna().sum()
)

Rows: 2912739
Missing magnitude before merge: 0
Missing magnitude after: 0
Missing event latitude: 0
Missing event longitude: 0


In [35]:
dataset[
    [
        "event_id",
        "magnitude",
        "event_lat",
        "event_lon"
    ]
].head()

,event_id,magnitude,event_lat,event_lon
0,us10002mft,5.5,36.68,71.3
1,us10002mft,5.5,36.68,71.3
2,us10002mft,5.5,36.68,71.3
3,us10002mft,5.5,36.68,71.3
4,us10002mft,5.5,36.68,71.3


In [36]:
dataset[
    [
        "event_id",
        "magnitude",
        "event_lat",
        "event_lon"
    ]
].head()

,event_id,magnitude,event_lat,event_lon
0,us10002mft,5.5,36.68,71.3
1,us10002mft,5.5,36.68,71.3
2,us10002mft,5.5,36.68,71.3
3,us10002mft,5.5,36.68,71.3
4,us10002mft,5.5,36.68,71.3


In [37]:
print(
    dataset[
        [
            "event_id",
            "magnitude",
            "event_lat",
            "event_lon"
        ]
    ].isna().sum()
)

event_id     0
magnitude    0
event_lat    0
event_lon    0
dtype: int64


In [38]:
def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):
    R = 6371.0

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = lat2 - lat1

    dlon = np.radians(
        lon2 - lon1
    )

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(
        np.sqrt(a)
    )

    return R * c

In [39]:
dataset["epicentral_distance_km"] = haversine_km(
    dataset["event_lat"],
    dataset["event_lon"],
    dataset["grid_lat"],
    dataset["grid_lon"]
)

display(
    dataset[
        [
            "event_id",
            "event_lat",
            "event_lon",
            "grid_lat",
            "grid_lon",
            "epicentral_distance_km"
        ]
    ].head()
)

,event_id,event_lat,event_lon,grid_lat,grid_lon,epicentral_distance_km
0,us10002mft,36.68,71.3,38.2842,69.3004,250.885571
1,us10002mft,36.68,71.3,38.2842,69.3171,249.851709
2,us10002mft,36.68,71.3,38.2842,69.3337,248.828424
3,us10002mft,36.68,71.3,38.2842,69.3504,247.803445
4,us10002mft,36.68,71.3,38.2842,69.3671,246.783004


In [51]:
# Find CSV files in the earthquake raw-data folder

print("Files in earthquake directory:\n")

for file in EARTHQUAKE_DIR.iterdir():
    print(file.name)

Files in earthquake directory:

india_region_earthquakes_2000_2026.csv
shakemap_available_events.csv


In [52]:
shakemap_events = pd.read_csv(
    EARTHQUAKE_DIR /
    "shakemap_available_events.csv"
)

print("Shape:", shakemap_events.shape)

print("\nColumns:")
print(shakemap_events.columns.tolist())

display(shakemap_events.head())

Shape: (635, 11)

Columns:
['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'place', 'type', 'status', 'net', 'id']


,time,latitude,longitude,depth,mag,magType,place,type,status,net,id
0,2000-01-03 22:34:12.640000+00:00,22.132,92.771,33.0,4.6,mb,"45 km SSW of Saiha, India",earthquake,reviewed,us,usp0009kqm
1,2000-01-05 09:45:18.710000+00:00,32.222,92.700,33.0,5.5,mwc,"102 km NE of Nagqu, China",earthquake,reviewed,us,usp0009ks9
2,2000-01-19 07:09:33.580000+00:00,36.372,70.379,206.9,6.0,mwb,"51 km ESE of Farkh?r, Afghanistan",earthquake,reviewed,us,usp0009mbv
3,2000-03-12 18:03:56.270000+00:00,17.099,73.672,33.0,5.0,mwc,"26 km SE of M?khjan, India",earthquake,reviewed,us,usp0009pmc
4,2000-03-31 14:13:31.470000+00:00,28.758,70.004,33.0,4.5,mb,"9 km NNE of Rojhan, Pakistan",earthquake,reviewed,us,usp0009qjf


In [53]:
print(
    "Total ShakeMap events:",
    len(shakemap_events)
)

print(
    "Missing depth:",
    shakemap_events["depth"].isna().sum()
)

display(
    shakemap_events[
        ["id", "depth", "mag", "latitude", "longitude"]
    ].head(20)
)

Total ShakeMap events: 635
Missing depth: 0


,id,depth,mag,latitude,longitude
0,usp0009kqm,33.0,4.6,22.132,92.771
1,usp0009ks9,33.0,5.5,32.222,92.700
2,usp0009mbv,206.9,6.0,36.372,70.379
3,usp0009pmc,33.0,5.0,17.099,73.672
4,usp0009qjf,33.0,4.5,28.758,70.004
5,usp0009qxd,10.0,4.9,17.147,73.637
6,usp0009sv4,107.7,6.3,35.975,70.657
7,usp0009ty9,33.0,6.0,28.723,65.383
8,usp0009u6j,33.0,6.3,26.856,97.238
9,usp0009wpm,141.4,6.3,36.283,70.924


In [54]:
depth_lookup = shakemap_events[
    [
        "id",
        "depth"
    ]
].copy()

depth_lookup = depth_lookup.rename(
    columns={
        "id": "event_id",
        "depth": "event_depth"
    }
)

# Keep only events present in our 40 ShakeMaps
depth_lookup = depth_lookup[
    depth_lookup["event_id"].isin(
        dataset["event_id"].unique()
    )
].copy()

print(
    "Events in lookup:",
    len(depth_lookup)
)

print(
    "Missing depths:",
    depth_lookup["event_depth"].isna().sum()
)

display(depth_lookup)

Events in lookup: 23
Missing depths: 0


,event_id,event_depth
391,usc000sy0y,66.00
392,usc000syca,6.61
394,usb000syze,21.63
403,us2000299v,13.61
410,us10002mft,191.00
412,us10002n5q,8.71
414,us20002z57,23.68
416,us100030qs,224.00
418,us100042n2,206.94
421,us10003vpz,8.00


In [55]:
dataset = dataset.drop(
    columns=["event_depth"],
    errors="ignore"
)

dataset = dataset.merge(
    depth_lookup,
    on="event_id",
    how="left"
)

print(
    "Total rows:",
    f"{len(dataset):,}"
)

print(
    "Missing depth:",
    dataset["event_depth"].isna().sum()
)

Total rows: 2,912,739
Missing depth: 1028330


In [56]:
event_metadata = (
    dataset[
        [
            "event_id",
            "magnitude",
            "event_depth",
            "event_lat",
            "event_lon"
        ]
    ]
    .drop_duplicates("event_id")
    .sort_values("magnitude", ascending=False)
)

print(
    "Unique events:",
    len(event_metadata)
)

display(event_metadata)

Unique events: 40


,event_id,magnitude,event_depth,event_lat,event_lon
207742,us10003vry,6.6,10.00,6.84,94.65
2567324,c000njrq,6.4,NaN,7.74,94.33
650521,us100088sf,6.0,10.00,6.15,92.30
2040095,b000qy82,6.0,NaN,18.20,88.04
452419,us100042n2,5.9,206.94,36.46,70.68
103389,us100030qs,5.9,224.00,36.53,71.21
1270805,us2000cp4g,5.8,10.00,8.25,91.77
2220312,b000ryuh,5.8,NaN,12.43,95.20
499173,us10004dtm,5.7,239.00,36.60,70.95
2754161,c000rff2,5.6,NaN,36.45,70.72


In [57]:
dataset["hypocentral_distance_km"] = np.sqrt(
    dataset["epicentral_distance_km"] ** 2
    +
    dataset["event_depth"] ** 2
)

display(
    dataset[
        [
            "epicentral_distance_km",
            "event_depth",
            "hypocentral_distance_km"
        ]
    ].describe()
)

,epicentral_distance_km,event_depth,hypocentral_distance_km
count,2.912739e+06,1.884409e+06,1.884409e+06
mean,2.154161e+02,4.334800e+01,2.501836e+02
std,1.137624e+02,6.445905e+01,1.176747e+02
min,4.203526e-02,6.610000e+00,6.625865e+00
25%,1.360624e+02,1.000000e+01,1.666315e+02
50%,1.938108e+02,1.000000e+01,2.338329e+02
75%,2.751437e+02,5.751000e+01,3.209529e+02
max,6.239846e+02,2.390000e+02,6.240647e+02


In [59]:
dataset = dataset.drop(
    columns=["event_depth"],
    errors="ignore"
)

dataset = dataset.merge(
    depth_lookup,
    on="event_id",
    how="left"
)

print("Total rows:", f"{len(dataset):,}")
print(
    "Missing depth:",
    dataset["event_depth"].isna().sum()
)

SyntaxError: '(' was never closed (2131321262.py, line 1)

In [60]:
# Build a unique event-level table from the current dataset

event_metadata = (
    dataset[
        [
            "event_id",
            "magnitude",
            "event_lat",
            "event_lon"
        ]
    ]
    .drop_duplicates("event_id")
    .copy()
)

print("ShakeMap events:", len(event_metadata))

ShakeMap events: 40


In [61]:
# Prepare original USGS catalog

catalog = earthquakes[
    [
        "id",
        "time",
        "latitude",
        "longitude",
        "depth",
        "mag"
    ]
].copy()

catalog = catalog.rename(
    columns={
        "id": "catalog_id",
        "latitude": "catalog_lat",
        "longitude": "catalog_lon",
        "depth": "catalog_depth",
        "mag": "catalog_mag"
    }
)

print("Catalog events:", len(catalog))

Catalog events: 18339


In [62]:
matches = []

for _, event in event_metadata.iterrows():

    candidates = catalog[
        (np.abs(
            catalog["catalog_lat"] -
            event["event_lat"]
        ) < 0.05)
        &
        (np.abs(
            catalog["catalog_lon"] -
            event["event_lon"]
        ) < 0.05)
        &
        (np.abs(
            catalog["catalog_mag"] -
            event["magnitude"]
        ) < 0.15)
    ].copy()

    if len(candidates) == 0:
        matches.append({
            "event_id": event["event_id"],
            "matched_catalog_id": None,
            "matched_depth": np.nan,
            "candidate_count": 0
        })
        continue

    # Choose closest spatial + magnitude match
    candidates["score"] = (
        np.abs(
            candidates["catalog_lat"] -
            event["event_lat"]
        )
        +
        np.abs(
            candidates["catalog_lon"] -
            event["event_lon"]
        )
        +
        np.abs(
            candidates["catalog_mag"] -
            event["magnitude"]
        )
    )

    best = candidates.sort_values(
        "score"
    ).iloc[0]

    matches.append({
        "event_id": event["event_id"],
        "matched_catalog_id": best["catalog_id"],
        "matched_depth": best["catalog_depth"],
        "candidate_count": len(candidates)
    })

depth_matches = pd.DataFrame(matches)

display(depth_matches)

,event_id,matched_catalog_id,matched_depth,candidate_count
0,us10002mft,us10002mft,191.00,1
1,us10002n5q,us10002n5q,8.71,1
2,us100030qs,us100030qs,224.00,1
3,us10003vpz,us10003vpz,8.00,1
4,us10003vry,us10003vry,10.00,1
5,us10003vsp,us10003vsp,10.00,1
6,us10003vxc,us10003vxc,10.00,1
7,us100042n2,us100042n2,206.94,2
8,us10004dtm,us10004dtm,239.00,1
9,us10004rhs,us10004rhs,174.00,3


In [63]:
print(
    "Total ShakeMap events:",
    len(depth_matches)
)

print(
    "Depths recovered:",
    depth_matches["matched_depth"].notna().sum()
)

print(
    "Still missing:",
    depth_matches["matched_depth"].isna().sum()
)

Total ShakeMap events: 40
Depths recovered: 40
Still missing: 0


In [64]:
# Remove the incomplete depth column
dataset = dataset.drop(
    columns=["event_depth"],
    errors="ignore"
)

# Rename recovered depth
depth_matches_clean = depth_matches[
    ["event_id", "matched_depth"]
].rename(
    columns={
        "matched_depth": "event_depth"
    }
)

# Merge
dataset = dataset.merge(
    depth_matches_clean,
    on="event_id",
    how="left"
)

print("Rows:", f"{len(dataset):,}")
print(
    "Missing depth:",
    dataset["event_depth"].isna().sum()
)

Rows: 2,912,739
Missing depth: 0


In [65]:
event_metadata = (
    dataset[
        [
            "event_id",
            "magnitude",
            "event_depth",
            "event_lat",
            "event_lon"
        ]
    ]
    .drop_duplicates("event_id")
    .sort_values(
        "magnitude",
        ascending=False
    )
)

print(
    "Unique events:",
    len(event_metadata)
)

display(event_metadata)

Unique events: 40


,event_id,magnitude,event_depth,event_lat,event_lon
207742,us10003vry,6.6,10.00,6.84,94.65
2567324,c000njrq,6.4,21.54,7.74,94.33
650521,us100088sf,6.0,10.00,6.15,92.30
2040095,b000qy82,6.0,47.23,18.20,88.04
452419,us100042n2,5.9,206.94,36.46,70.68
103389,us100030qs,5.9,224.00,36.53,71.21
1270805,us2000cp4g,5.8,10.00,8.25,91.77
2220312,b000ryuh,5.8,10.00,12.43,95.20
499173,us10004dtm,5.7,239.00,36.60,70.95
2754161,c000rff2,5.6,200.00,36.45,70.72


In [66]:
dataset["hypocentral_distance_km"] = np.sqrt(
    dataset["epicentral_distance_km"] ** 2
    +
    dataset["event_depth"] ** 2
)

In [67]:
display(
    dataset[
        [
            "magnitude",
            "event_depth",
            "epicentral_distance_km",
            "hypocentral_distance_km"
        ]
    ].describe()
)

,magnitude,event_depth,epicentral_distance_km,hypocentral_distance_km
count,2.912739e+06,2.912739e+06,2.912739e+06,2.912739e+06
mean,5.594578e+00,4.040300e+01,2.154161e+02,2.288580e+02
std,3.827159e-01,5.766262e+01,1.137624e+02,1.092124e+02
min,5.100000e+00,6.610000e+00,4.203526e-02,6.625865e+00
25%,5.300000e+00,1.000000e+01,1.360624e+02,1.529822e+02
50%,5.500000e+00,1.300000e+01,1.938108e+02,2.137442e+02
75%,5.800000e+00,4.723000e+01,2.751437e+02,2.852601e+02
max,6.600000e+00,2.390000e+02,6.239846e+02,6.240647e+02


In [68]:
print(
    "Missing epicentral distance:",
    dataset["epicentral_distance_km"].isna().sum()
)

print(
    "Missing hypocentral distance:",
    dataset["hypocentral_distance_km"].isna().sum()
)

Missing epicentral distance: 0
Missing hypocentral distance: 0


In [69]:
target_columns = [
    "PGA",
    "PGV",
    "MMI",
    "PSA03",
    "PSA10",
    "PSA30"
]

target_stats = dataset[
    target_columns
].describe().T

display(target_stats)

,count,mean,std,min,25%,50%,75%,max
PGA,2912739.0,0.596749,1.239956,0.00,0.10,0.28,0.69,90.17
PGV,2912739.0,0.398947,0.739707,0.01,0.10,0.21,0.46,41.44
MMI,2912739.0,2.607086,0.735919,1.00,2.21,2.65,3.08,7.74
PSA03,2912739.0,1.226700,2.264547,0.01,0.26,0.68,1.50,139.19
PSA10,2912739.0,0.441829,0.748941,0.01,0.13,0.24,0.51,45.52
PSA30,2912739.0,0.098189,0.158961,0.00,0.03,0.05,0.10,7.70


In [70]:
target_missingness = pd.DataFrame({
    "missing_count": dataset[target_columns].isna().sum(),
    "missing_percent": (
        dataset[target_columns].isna().mean() * 100
    )
})

display(target_missingness)

,missing_count,missing_percent
PGA,0,0.0
PGV,0,0.0
MMI,0,0.0
PSA03,0,0.0
PSA10,0,0.0
PSA30,0,0.0


In [71]:
for col in target_columns:

    print(f"\n===== {col} =====")

    print(
        "Negative:",
        (dataset[col] < 0).sum()
    )

    print(
        "Zero:",
        (dataset[col] == 0).sum()
    )

    print(
        "NaN:",
        dataset[col].isna().sum()
    )


===== PGA =====
Negative: 0
Zero: 39370
NaN: 0

===== PGV =====
Negative: 0
Zero: 0
NaN: 0

===== MMI =====
Negative: 0
Zero: 0
NaN: 0

===== PSA03 =====
Negative: 0
Zero: 0
NaN: 0

===== PSA10 =====
Negative: 0
Zero: 0
NaN: 0

===== PSA30 =====
Negative: 0
Zero: 66243
NaN: 0


In [72]:
integrated_file = (
    INTERIM_DIR /
    "earthquake_ground_motion_integrated.csv"
)

dataset.to_csv(
    integrated_file,
    index=False
)

print("Saved:")
print(integrated_file)

print(
    "\nRows:",
    f"{len(dataset):,}"
)

print(
    "Events:",
    dataset["event_id"].nunique()
)

Saved:
C:\Documents\Earthquake_ML_DL\data\interim\earthquake_ground_motion_integrated.csv

Rows: 2,912,739
Events: 40
